# SPS Self-Specialization — Demo Test

A simple Colab test of the deterministic Stage 1 self-specialization prototype.

Flow: clone → install → simple tests → boundary rules → manually add S0 → inspect capabilities → request float specialization → inspect new capability → manually execute FloatMultiplication.

## 1. Clone latest `main` and install dependencies

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip install -q -r requirements.txt pytest
!git rev-parse --short HEAD

## 2. Run simple tests

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 3. Check allowed/rejected growth rules

In [ ]:
from specialization.type_specialization_rules import is_type_specialization_allowed

allowed = is_type_specialization_allowed(
    'multiply', ['int', 'int'], 'int',
    'multiply', ['float', 'float'], 'float'
)
rejected = is_type_specialization_allowed(
    'multiply', ['int', 'int'], 'int',
    'add', ['int', 'int'], 'int'
)

print('int multiplication -> float multiplication:', 'ALLOWED' if allowed else 'REJECTED')
print('multiplication -> addition:', 'ALLOWED' if rejected else 'REJECTED')
assert allowed
assert not rejected
print('Boundary checks passed.')

## 4. Manually add the initial S0 capability

In [ ]:
import shutil
from specialization import Capability, CapabilityRegistry

storage = CapabilityRegistry.default_storage_dir()
if storage.exists():
    shutil.rmtree(storage)

registry = CapabilityRegistry.persistent()
INTEGER_SOURCE = '''def execute(a: int, b: int) -> int:\n    return a * b\n'''

integer_mul = Capability.create(
    'IntegerMultiplication',
    '1.0',
    'S0',
    ['int', 'int'],
    'int',
    INTEGER_SOURCE,
)
registry.register(integer_mul)

print('Manually added:')
print(f'  {integer_mul.name} [{integer_mul.state}]')
print('6 * 7 =', integer_mul.execute(6, 7))

## 5. Check capabilities before specialization

In [ ]:
print(f'Total capabilities: {len(registry.all())}')
for cap in registry.all():
    print(f'  {cap.name} [{cap.state}] | input={cap.input_types} | output={cap.output_type}')

assert [cap.name for cap in registry.all()] == ['IntegerMultiplication']

## 6. Manually request float multiplication

This triggers the specialization mechanism. The request is executed through the dispatcher; no AI model is used.

In [ ]:
from specialization import CapabilityDispatcher, EvolutionEngine, Verifier

dispatcher = CapabilityDispatcher(registry, EvolutionEngine(registry, Verifier()))

value, float_mul = dispatcher.execute(
    'multiply',
    2.5,
    4.0,
    ('FloatMultiplication', 'float', [(2.5, 4.0, 10.0), (-2.5, 4.0, -10.0)]),
)

print(f'Created: {float_mul.name} [{float_mul.state}]')
print('Result from generated capability:', value)

## 7. Check capabilities after specialization

In [ ]:
print(f'Total capabilities: {len(registry.all())}')
for cap in sorted(registry.all(), key=lambda c: (c.created_at, c.id)):
    print(f'  {cap.name} [{cap.state}] | input={cap.input_types} | output={cap.output_type} | parent={cap.parent_id}')

assert registry.get('IntegerMultiplication').state == 'S0'
assert registry.get('FloatMultiplication').state == 'S1'

## 8. Manually execute the new FloatMultiplication capability

In [ ]:
float_cap = registry.get('FloatMultiplication')
print('FloatMultiplication [S1]')
print('3.5 * 2.0 =', float_cap.execute(3.5, 2.0))
print('10.0 * 1.5 =', float_cap.execute(10.0, 1.5))

## 9. Final capability structure

In [ ]:
general = registry.get(float_cap.parent_id)
print(f'{general.name} [{general.state}]')
for child in registry.children(general.id):
    print(f'  └── {child.name} [{child.state}]')

print('\nGenerated source code:')
print(float_cap.source_code)